# Exploratory Loss Analysis

This notebook explores a fully synthetic personal auto comprehensive coverage dataset. It is designed as a public portfolio artifact and does not use proprietary data, internal table names, confidential business rules, or employer-specific terminology.

## Business Problem

Product, actuarial, and underwriting stakeholders need a repeatable way to monitor loss performance, identify deteriorating segments, and separate stable trends from small-sample noise or catastrophe-driven volatility.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from generate_synthetic_data import main as generate_data
from data_processing import load_data, prepare_monthly_summary, prepare_segment_summary, apply_credibility_filter
from metrics import add_core_metrics

In [ ]:
data_path = PROJECT_ROOT / 'data' / 'synthetic_auto_claims.csv'
if not data_path.exists():
    generate_data()

df = load_data(data_path)
df.head()

## Data Dictionary

| Column | Description |
| --- | --- |
| policy_id | Synthetic policy identifier |
| vehicle_id | Synthetic vehicle identifier |
| period_month | Monthly accounting period |
| state_code | Generic U.S. state abbreviation |
| region | Broad geographic region |
| sales_channel | Generic distribution channel |
| business_type | New business or renewal |
| vehicle_make / vehicle_model | Fictional vehicle segment |
| coverage_type | Comprehensive coverage |
| loss_category | Synthetic claim category |
| catastrophe_flag | Synthetic catastrophe indicator |
| exposure_units | Earned exposure units |
| earned_premium | Synthetic earned premium |
| on_level_premium | Synthetic adjusted premium basis |
| claim_count | Synthetic claim count |
| incurred_loss | Synthetic incurred loss amount |

In [ ]:
df.shape, df.dtypes

## Core Metrics

The key monitoring metrics are loss ratio, claim frequency, and severity. Loss ratio can be calculated on earned premium or on-level premium depending on the analytical question.

In [ ]:
overall = add_core_metrics(
    df.agg({
        'exposure_units': 'sum',
        'earned_premium': 'sum',
        'on_level_premium': 'sum',
        'claim_count': 'sum',
        'incurred_loss': 'sum'
    }).to_frame().T,
    premium_basis='earned_premium'
)
overall[['earned_premium', 'incurred_loss', 'claim_count', 'loss_ratio', 'claim_frequency', 'severity']]

## Monthly Loss Ratio Trend

In [ ]:
monthly = prepare_monthly_summary(df, premium_basis='earned_premium')
monthly[['period_month', 'earned_premium', 'incurred_loss', 'claim_count', 'loss_ratio']].head()

In [ ]:
monthly.sort_values('loss_ratio', ascending=False).head(10)

## Catastrophe vs Non-Catastrophe Experience

In [ ]:
cat_summary = prepare_segment_summary(df, ['catastrophe_flag'], premium_basis='earned_premium')
cat_summary[['catastrophe_flag', 'earned_premium', 'incurred_loss', 'claim_count', 'loss_ratio', 'severity']]

## Credibility-Filtered Vehicle Segment Ranking

In [ ]:
vehicle_segments = prepare_segment_summary(df, ['vehicle_make', 'vehicle_model'], premium_basis='earned_premium')
credible_vehicle_segments = apply_credibility_filter(vehicle_segments, min_claim_count=25)
credible_vehicle_segments.sort_values('loss_ratio', ascending=False).head(10)

## Business Type and Sales Channel Performance

In [ ]:
business_summary = prepare_segment_summary(df, ['business_type'], premium_basis='earned_premium')
channel_summary = prepare_segment_summary(df, ['sales_channel'], premium_basis='earned_premium')
business_summary

In [ ]:
channel_summary.sort_values('claim_frequency', ascending=False)

## Conclusion

This synthetic dataset supports a realistic monitoring workflow: identify headline loss ratio movement, isolate catastrophe-driven volatility, rank credible vehicle segments, and compare business mix across channels and new versus renewal business. The same pattern can be extended to other coverages or product lines by standardizing dimensions, metric definitions, and credibility rules.